<a href="https://colab.research.google.com/github/kassimsesay7-rgb/AAIPP/blob/main/DistilBERT_Lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### We will use the Wikitext-2 dataset, which is a standard benchmark dataset for language modeling.

#####Project Structure
* Objective: Fine-tune DistilBERT to fill in missing words (masks) in a sentence.

* Dataset: wikitext-2-raw-v1 (via Hugging Face).

* Model: distilbert-base-uncased.

### Step 1: Set Up Environment and Load Data
First, we install the necessary libraries and load the dataset.

In [ ]:
# Install necessary libraries (Run this in a cell)
!pip install transformers datasets evaluate torch

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Load the Wikitext-2 dataset
# We use the 'raw' version to handle preprocessing ourselves.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

# Inspect the dataset
print(f"Train dataset size: {len(dataset['train'])}")
print(f"Test dataset size: {len(dataset['test'])}")
print("Sample text:", dataset['train'][10]['text'])

### Step 2: Preprocess Dataset (Deliverable A)
We need to tokenize the text and group it into chunks that fit the model's context window (usually 512 tokens, but we will use 128 for faster training in this demo).

In [ ]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    # We allow truncation but no padding yet; padding happens dynamically in the collator
    return tokenizer(examples["text"], return_special_tokens_without_tokenizing=True)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])

# Function to group texts into chunks of block_size
block_size = 128

def group_texts(examples):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])

    # We drop the small remainder, we could add padding if the model supported it instead of this drop
    total_length = (total_length // block_size) * block_size

    # Split by chunks of max_len
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# Apply the grouping function
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

print("Preprocessing complete. Data shape:", lm_datasets)

## Step 3 & 4: Create Masked Data Collator & Load Model
We use a special data collator that randomly masks tokens (standard is 15% probability) during training. This prevents the model from just memorizing the data.

In [ ]:
from transformers import DataCollatorForLanguageModeling, DistilBertForMaskedLM

# 3. Create Masked Data Collator
# mlm_probability=0.15 is the standard BERT masking rate
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

# 4. Load Pre-trained DistilBERT Model
model = DistilBertForMaskedLM.from_pretrained(model_checkpoint)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded on cuda


## Step 5: Train Model and Evaluation (Deliverable B & C)
We use the Hugging Face Trainer API. We will use a smaller subset for the sake of demonstration speed, but .select() can be removed to train on the full dataset.

Metric: We calculate Perplexity. Lower perplexity means the model is less "surprised" by the text and understands it better.

In [ ]:
from transformers import Trainer, TrainingArguments
import math

# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./distilbert-mlm-wikitext",
    eval_strategy="epoch", # Changed from evaluation_strategy
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=False,
    per_device_train_batch_size=16, # Adjust based on your GPU VRAM
    per_device_eval_batch_size=16,
    logging_steps=50,
    fp16=torch.cuda.is_available(), # Enable mixed precision if using GPU
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"], # Use .select(range(1000)) here for quick debug
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator
)

# 5. Train the Model
trainer.train()

# Evaluate the model
eval_results = trainer.evaluate()

# Calculate Perplexity (e^loss)
perplexity = math.exp(eval_results['eval_loss'])

print(f"--- Evaluation Results ---")
print(f"Perplexity: {perplexity:.2f}")
print(f"Evaluation Loss: {eval_results['eval_loss']:.4f}")

# Save the model artifacts (Deliverable B)
trainer.save_model("./distilbert-mlm-final")

Epoch,Training Loss,Validation Loss
1,2.088700,1.999129
2,2.057726,1.983720
3,2.051105,1.955089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--- Evaluation Results ---
Perplexity: 6.97
Evaluation Loss: 1.9421


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Step 6: Inference Pipeline (Deliverable D)
Now that the model is fine-tuned, we can use it to fill in masks in new sentences.

In [ ]:
from transformers import pipeline

# Load the fine-tuned model for inference
mask_filler = pipeline(
    "fill-mask",
    model="./distilbert-mlm-final",
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# Test sentences
text1 = "The capital of France is [MASK]."
text2 = "Mathematical formulas are often written using [MASK]."

preds1 = mask_filler(text1)
preds2 = mask_filler(text2)

print(f"\nQuery: {text1}")
for pred in preds1:
    print(f"  {pred['token_str']} (Score: {pred['score']:.4f})")

print(f"\nQuery: {text2}")
for pred in preds2:
    print(f"  {pred['token_str']} (Score: {pred['score']:.4f})")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


Query: The capital of France is [MASK].
  paris (Score: 0.3243)
  marseille (Score: 0.2771)
  toulouse (Score: 0.0463)
  lyon (Score: 0.0386)
  lille (Score: 0.0368)

Query: Mathematical formulas are often written using [MASK].
  symbols (Score: 0.1351)
  parentheses (Score: 0.0461)
  computers (Score: 0.0384)
  brackets (Score: 0.0281)
  latin (Score: 0.0199)
